# Lesson 1: SDK Setup

See [docs/01-sdk-setup.md](../docs/01-sdk-setup.md).

## What is going on?

There are **two different kinds of deployment** in SAP AI Launchpad:

1. **Direct model deployment** from **Generative AI Hub -> Model Library**
   - Use this with `gen_ai_hub.proxy.native.openai.*`
   - Most reliable input: `deployment_id=...`
   - You can get that from the deployment URL shown in Launchpad

2. **Orchestration deployment** from **ML Operations -> Deployments** for scenario `orchestration`
   - Use this with `OrchestrationService`
   - This is what `ORCH_DEPLOYMENT_URL` is for

If `model_name='gpt-4.1-mini'` fails, that usually means the exact model name is not what the SDK sees for your deployment. Using the **deployment URL / deployment ID** from Launchpad is safer.

In [1]:
import os
import sys
sys.path.insert(0, '..')

from src.init_env import set_environment_variables
config = set_environment_variables()
print({
    'AICORE_AUTH_URL': os.environ.get('AICORE_AUTH_URL'),
    'AICORE_RESOURCE_GROUP': os.environ.get('AICORE_RESOURCE_GROUP'),
    'AICORE_BASE_URL': os.environ.get('AICORE_BASE_URL'),
    'ORCH_DEPLOYMENT_URL': os.environ.get('ORCH_DEPLOYMENT_URL'),
})

{'AICORE_AUTH_URL': 'https://dev-tools-sbpa-sokd3vuh.authentication.ap10.hana.ondemand.com/oauth/token', 'AICORE_RESOURCE_GROUP': 'default', 'AICORE_BASE_URL': 'https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2', 'ORCH_DEPLOYMENT_URL': 'https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2/inference/deployments/dc829c4504a1fd33'}


In [2]:
%pip install -q -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from src.deployments import list_running_deployments

deployments, proxy = list_running_deployments()

RUNNING deployments in resource group 'default':
  - id=d665c62123b5840e  scenario=foundation-models  config=gpt-4.1-mini_autogenerated  url=https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2/inference/deployments/d665c62123b5840e
  - id=dc79261eee5a890c  scenario=foundation-models  config=gemini-2.5-flash_autogenerated  url=https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2/inference/deployments/dc79261eee5a890c
  - id=dc829c4504a1fd33  scenario=orchestration  config=defaultOrchestrationConfig  url=https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2/inference/deployments/dc829c4504a1fd33

If you have a model deployment open in Launchpad, copy its URL into DIRECT_DEPLOYMENT_URL in config/config.json.


In [5]:
from urllib.parse import urlparse

# Preferred: store DIRECT_DEPLOYMENT_URL / ID in config/config.json (gitignored)
# under keys DIRECT_DEPLOYMENT_URL and DIRECT_DEPLOYMENT_ID. They are loaded
# into environment variables by src.init_env.set_environment_variables().

DIRECT_DEPLOYMENT_URL = os.environ.get("DIRECT_DEPLOYMENT_URL", "")
DIRECT_DEPLOYMENT_ID = os.environ.get("DIRECT_DEPLOYMENT_ID", "")

if not DIRECT_DEPLOYMENT_URL and not DIRECT_DEPLOYMENT_ID:
    print(
        "Set DIRECT_DEPLOYMENT_URL (or DIRECT_DEPLOYMENT_ID) in config/config.json "
        "or as an environment variable. Then rerun the first cell and this one."
    )
else:
    if not DIRECT_DEPLOYMENT_ID and DIRECT_DEPLOYMENT_URL:
        DIRECT_DEPLOYMENT_ID = urlparse(DIRECT_DEPLOYMENT_URL).path.rstrip("/").split("/")[-1]
    print("DIRECT_DEPLOYMENT_ID =", DIRECT_DEPLOYMENT_ID)

DIRECT_DEPLOYMENT_ID = d665c62123b5840e


In [6]:
from gen_ai_hub.proxy.native.openai import completions

if not DIRECT_DEPLOYMENT_URL:
    print("Set DIRECT_DEPLOYMENT_URL in the cell above first.")
else:
    response = completions.create(
        deployment_id=DIRECT_DEPLOYMENT_ID,
        prompt='The Answer to the Ultimate Question of Life, the Universe, and Everything is',
        max_tokens=7,
        temperature=0,
    )
    print(response.choices[0].text if hasattr(response.choices[0], 'text') else response)

BadRequestError: Error code: 400 - {'error': {'code': 'OperationNotSupported', 'message': 'The completion operation does not work with the specified model, gpt-4.1-mini. Please choose different model and try again. You can learn more about which models can be used with each operation here: https://go.microsoft.com/fwlink/?linkid=2197993.'}}

In [ ]:
from gen_ai_hub.proxy.native.openai import chat

if not DIRECT_DEPLOYMENT_URL:
    print("Set DIRECT_DEPLOYMENT_URL in the cell above first.")
else:
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'What is SAP Generative AI Hub in one sentence?'},
    ]
    response = chat.completions.create(
        deployment_id=DIRECT_DEPLOYMENT_ID,
        model_name='gpt-4o'
        messages=messages,
    )
    print(response.choices[0].message.content)

SAP Generative AI Hub is a centralized platform by SAP that integrates generative AI technologies to enable businesses to create, manage, and deploy AI-driven applications and solutions efficiently.


## Orchestration check (separate from direct model deployment)

Use this only to verify your orchestration deployment from `ORCH_DEPLOYMENT_URL`. This is a different path from the direct OpenAI-like proxy above.

In [ ]:
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.service import OrchestrationService

if not os.environ.get('ORCH_DEPLOYMENT_URL'):
    print('No ORCH_DEPLOYMENT_URL set in config.json')
else:
    service = OrchestrationService(api_url=os.environ['ORCH_DEPLOYMENT_URL'], proxy_client=proxy)
    config = OrchestrationConfig(
        template=Template(messages=[
            SystemMessage('You are a helpful assistant.'),
            UserMessage('{{?question}}'),
        ]),
        llm=LLM(name='gpt-4.1', parameters={'max_tokens': 100, 'temperature': 0}),
    )
    result = service.run(
        config=config,
        template_values=[TemplateValue(name='question', value='What is SAP Generative AI Hub in one sentence?')],
    )
    print(result.orchestration_result.choices[0].message.content)